In [1]:
from __future__ import annotations

import os
import math
import time
import textwrap
import warnings
import pickle
from pathlib import Path
from collections import defaultdict
from typing import Dict, Tuple, Iterable, Optional, List

import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point, LineString, box, MultiPolygon
from tqdm import tqdm

import networkx as nx
import osmnx as ox

from sklearn.metrics import roc_auc_score
from statsmodels.genmod.generalized_linear_model import GLM
from statsmodels.genmod.families import NegativeBinomial
import statsmodels.api as sm
import requests

try:
    from shapely.validation import make_valid  # shapely 2.x
except Exception:
    make_valid = None

warnings.filterwarnings("ignore", category=UserWarning)

# -----------------------------------------------------------------------------
# SETTINGS & CACHE
# -----------------------------------------------------------------------------
ox.settings.log_console = True
ox.settings.use_cache = True
ox.settings.timeout = 180  # seconds

CACHE_DIR = Path("cache_nyc_ulmm")
CACHE_DIR.mkdir(parents=True, exist_ok=True)
REBUILD = False

def cache_path(name: str) -> Path:
    return CACHE_DIR / f"{name}.pkl"

def save_obj(obj, name: str):
    path = cache_path(name)
    with open(path, "wb") as f:
        pickle.dump(obj, f, protocol=pickle.HIGHEST_PROTOCOL)
    print(f"[cache] saved -> {path}")

def load_obj(name: str):
    path = cache_path(name)
    if not REBUILD and path.exists():
        with open(path, "rb") as f:
            obj = pickle.load(f)
        print(f"[cache] loaded <- {path}")
        return obj
    return None

# -----------------------------------------------------------------------------
# CONFIG
# -----------------------------------------------------------------------------
DATE_FROM = "2024-01-01"
DATE_TO   = "2024-12-31"

BUSINESS_HOURS = (9, 17)

SOCRATA_LIMIT = 50000
SOCRATA_APP_TOKEN = os.getenv("SOCRATA_APP_TOKEN", "")

LAMBDA_CURB = 0.5

GRID_SIZE_M = 500
OSM_POI_TAGS = {
    "shop": True,
    "amenity": ["cafe","restaurant","fast_food","bar","pub","pharmacy","bank",
                "cinema","clinic","doctors","library","marketplace","biergarten",
                "ice_cream","post_office","parking"]
}

OSM_ACCESS_TAGS = [
    {"amenity": "post_office"},
    {"amenity": "post_box"},
    {"amenity": "parcel_locker"},
    {"amenity": "vending_machine", "vending": "parcel_locker"},
]

ACCESS_ACTIVITY_RADIUS_M = 50
EVENT_TO_EDGE_MAXDIST_M  = 30

SERVC_DELTA = 1

RNG = np.random.default_rng(12345)

EPS_DENOM = 1e-6  # numerical safety for divisions

# -----------------------------------------------------------------------------
# OSMnx / GeoPandas compatibility helpers
# -----------------------------------------------------------------------------
def ox_features_from_polygon(poly, tags: dict) -> gpd.GeoDataFrame:
    """OSMnx 2.x/1.x compatible features query; returns empty GDF if nothing found."""
    # OSMnx 2.x
    try:
        return ox.features_from_polygon(poly, tags)
    except AttributeError:
        pass
    # OSMnx 1.x
    try:
        return ox.geometries_from_polygon(poly, tags)
    except AttributeError:
        pass
    # Direct import fallback
    try:
        from osmnx.features import features_from_polygon
        return features_from_polygon(poly, tags)
    except Exception as e:
        raise RuntimeError("No compatible features/geometries_from_polygon in this OSMnx version") from e

def ox_add_edge_speeds(G):
    try:
        return ox.add_edge_speeds(G)
    except AttributeError:
        from osmnx.speed import add_edge_speeds
        return add_edge_speeds(G)

def ox_add_edge_travel_times(G):
    try:
        return ox.add_edge_travel_times(G)
    except AttributeError:
        from osmnx.speed import add_edge_travel_times
        return add_edge_travel_times(G)

def centroid_wgs84_safe(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    """Centroids in projected CRS then back to WGS84; robust for mixed geoms."""
    if gdf.empty:
        return gdf
    gdf = gdf[~gdf.geometry.is_empty & gdf.geometry.notna()].copy()
    try:
        local_crs = gdf.estimate_utm_crs()
        gdfp = gdf.to_crs(local_crs)
        gdfp["geometry"] = gdfp.geometry.centroid
        return gdfp.to_crs("EPSG:4326").set_geometry("geometry")
    except Exception:
        g = gdf.copy()
        g["geometry"] = g.geometry.representative_point()
        if g.crs is None:
            g.set_crs("EPSG:4326", inplace=True)
        return g.set_geometry("geometry")

# -----------------------------------------------------------------------------
# Socrata helpers
# -----------------------------------------------------------------------------
def socrata_get(url: str, params: dict) -> List[dict]:
    out = []
    offset = 0
    headers = {}
    if SOCRATA_APP_TOKEN:
        headers["X-App-Token"] = SOCRATA_APP_TOKEN

    while True:
        q = params.copy()
        q["$limit"] = SOCRATA_LIMIT
        q["$offset"] = offset
        resp = requests.get(url, params=q, headers=headers, timeout=120)
        resp.raise_for_status()
        rows = resp.json()
        if not rows:
            break
        out.extend(rows)
        if len(rows) < SOCRATA_LIMIT:
            break
        offset += SOCRATA_LIMIT
    return out

# -----------------------------------------------------------------------------
# NYC BOUNDARY & STREET GRAPH
# -----------------------------------------------------------------------------
def get_nyc_boundary_wgs84() -> gpd.GeoSeries:
    cached = load_obj("nyc_boundary_wgs84")
    if cached is not None:
        return cached

    g = ox.geocode_to_gdf("New York City, New York, USA")  # WGS84 by default
    geom = g.geometry.iloc[0]
    if isinstance(geom, MultiPolygon):
        geom = max(geom.geoms, key=lambda p: p.area)
    if not geom.is_valid:
        if make_valid is not None:
            geom = make_valid(geom)
            if isinstance(geom, MultiPolygon):
                geom = max(geom.geoms, key=lambda p: p.area)
        else:
            geom = geom.buffer(0)
    if isinstance(geom, MultiPolygon):
        geom = max(geom.geoms, key=lambda p: p.area)
    out = gpd.GeoSeries([geom], crs="EPSG:4326")
    save_obj(out, "nyc_boundary_wgs84")
    return out

def build_curb_aware_graph(nyc_poly_wgs84) -> Tuple[nx.MultiDiGraph, gpd.GeoDataFrame, gpd.GeoDataFrame]:
    G = load_obj("graph_G")
    nodes_gdf = load_obj("graph_nodes_gdf")
    edges_gdf = load_obj("graph_edges_gdf")
    if G is not None and nodes_gdf is not None and edges_gdf is not None:
        return G, nodes_gdf, edges_gdf

    print("Downloading directed drive graph…")
    polygon = nyc_poly_wgs84.iloc[0] if isinstance(nyc_poly_wgs84, gpd.GeoSeries) else nyc_poly_wgs84
    try:
        if not polygon.is_valid:
            if make_valid:
                polygon = make_valid(polygon)
                if isinstance(polygon, MultiPolygon):
                    polygon = max(polygon.geoms, key=lambda p: p.area)
            else:
                polygon = polygon.buffer(0)
    except Exception:
        pass

    # Use drive_service to keep service/aisle links that matter at the curb
    G = ox.graph_from_polygon(polygon, network_type="drive_service", simplify=True)
    G = ox_add_edge_speeds(G)
    G = ox_add_edge_travel_times(G)

    # Simple curb/friction proxy phi in [0,1]
    lane_w, restrict_w, motorway_w = 0.5, 0.3, 0.2
    for _, _, _, d in G.edges(keys=True, data=True):
        lanes = d.get("lanes")
        try:
            lanes_val = float(lanes) if lanes is not None else 1.0
        except Exception:
            lanes_val = 1.0
        f_lane = 1.0 if lanes_val <= 1 else 0.0

        access = str(d.get("access", "")).lower()
        service = str(d.get("service", "")).lower()
        f_restrict = 1.0 if access in ("private", "no") or service in ("driveway", "parking_aisle") else 0.0

        highway = str(d.get("highway", "")).lower()
        f_motorway = 1.0 if highway in ("motorway", "trunk") else 0.0

        phi = lane_w * f_lane + restrict_w * f_restrict + motorway_w * f_motorway
        d["phi"] = float(phi)

        ff = float(d.get("travel_time", 0.0))
        d["tt"] = ff * (1.0 + LAMBDA_CURB * d["phi"])

    nodes_gdf, edges_gdf = ox.graph_to_gdfs(G)
    nodes_gdf = nodes_gdf.to_crs(3857).reset_index()
    edges_gdf = edges_gdf.to_crs(3857).reset_index()

    save_obj(G, "graph_G")
    save_obj(nodes_gdf, "graph_nodes_gdf")
    save_obj(edges_gdf, "graph_edges_gdf")
    return G, nodes_gdf, edges_gdf

# -----------------------------------------------------------------------------
# DEMAND GRID & WEIGHTS FROM OSM POIs
# -----------------------------------------------------------------------------
def build_demand_grid(nyc_poly_3857: gpd.GeoSeries) -> gpd.GeoDataFrame:
    cached = load_obj("demand_grid")
    if cached is not None:
        return cached

    minx, miny, maxx, maxy = nyc_poly_3857.total_bounds
    xs = np.arange(minx, maxx, GRID_SIZE_M)
    ys = np.arange(miny, maxy, GRID_SIZE_M)
    cells = [box(x, y, x + GRID_SIZE_M, y + GRID_SIZE_M) for x in xs for y in ys]
    grid = gpd.GeoDataFrame(geometry=cells, crs=3857)

    mask = gpd.GeoDataFrame(geometry=nyc_poly_3857, crs=3857)
    grid = gpd.overlay(grid, mask, how="intersection")
    grid["d_id"] = np.arange(len(grid))

    save_obj(grid, "demand_grid")
    return grid

def download_osm_pois(nyc_poly_wgs84) -> gpd.GeoDataFrame:
    cached = load_obj("pois")
    if cached is not None:
        return cached

    print("Downloading OSM POIs for demand weights…")
    polygon = nyc_poly_wgs84.iloc[0] if isinstance(nyc_poly_wgs84, gpd.GeoSeries) else nyc_poly_wgs84
    layers = []
    for key, val in OSM_POI_TAGS.items():
        q = {key: True} if val is True else {key: val}
        try:
            g = ox_features_from_polygon(polygon, q)
            if not g.empty:
                layers.append(g)
        except Exception as e:
            print(f"  POI tag {q} skipped: {e}")

    if not layers:
        pois = gpd.GeoDataFrame(geometry=[], crs="EPSG:4326").to_crs(3857)
        save_obj(pois, "pois")
        return pois

    pois = pd.concat(layers, ignore_index=True)
    pois = pois[~pois.geometry.isna()].copy()
    pois = centroid_wgs84_safe(pois).to_crs(3857)  # safe centroids
    pois = pois[["geometry"]].reset_index(drop=True)

    save_obj(pois, "pois")
    return pois

def weight_demand_cells(grid_gdf, pois_gdf) -> gpd.GeoDataFrame:
    cached = load_obj("demand_weighted")
    if cached is not None:
        return cached

    grid = grid_gdf.copy()
    if pois_gdf.empty:
        grid["w_d"] = 1.0
        save_obj(grid, "demand_weighted")
        return grid

    joined = gpd.sjoin(pois_gdf[["geometry"]], grid[["d_id","geometry"]], how="left", predicate="within")
    counts = joined.groupby("d_id").size().rename("poi_count")
    grid = grid.merge(counts, on="d_id", how="left").fillna({"poi_count": 0})
    grid["w_d"] = grid["poi_count"].astype(float) + 1.0

    save_obj(grid, "demand_weighted")
    return grid

# -----------------------------------------------------------------------------
# ACCESS POINTS & ANCHORS
# -----------------------------------------------------------------------------
def download_access_points(nyc_poly_wgs84) -> gpd.GeoDataFrame:
    cached = load_obj("access_points")
    if cached is not None:
        return cached

    print("Downloading access points (post offices & lockers)…")
    polygon = nyc_poly_wgs84.iloc[0] if isinstance(nyc_poly_wgs84, gpd.GeoSeries) else nyc_poly_wgs84
    layers = []

    def _try(getter, where, tag):
        try:
            g = getter(where, tags=tag)
            if not g.empty:
                layers.append(g)
        except Exception:
            pass

    for tag in OSM_ACCESS_TAGS:
        _try(ox.features_from_polygon, polygon, tag)

    if not layers:
        for tag in OSM_ACCESS_TAGS:
            _try(ox.features_from_place, "New York City, New York, USA", tag)

    if not layers:
        acc = gpd.GeoDataFrame(geometry=[], crs=4326).to_crs(3857)
        save_obj(acc, "access_points")
        return acc

    acc = pd.concat(layers, ignore_index=True)
    acc = acc[~acc.geometry.isna()].copy()
    acc = centroid_wgs84_safe(acc).to_crs(3857)
    acc = acc.reset_index(drop=True)
    acc["a_id"] = range(len(acc))
    acc["alpha"] = 1.0
    acc = acc[["a_id","alpha","geometry"]]

    save_obj(acc, "access_points")
    return acc

def anchor_points_to_graph(nodes_gdf, points_gdf, label_col) -> pd.DataFrame:
    """points_gdf and nodes_gdf must be in 3857. Returns: a_id, node(osmid), dist, label."""
    cached_name = f"anchor_{label_col}"
    cached = load_obj(cached_name)
    if cached is not None:
        return cached

    if points_gdf.empty:
        out = pd.DataFrame(columns=["a_id","node","dist","label"])
        save_obj(out, cached_name)
        return out

    nodes_reset = nodes_gdf.reset_index()[["osmid", "geometry"]]
    nn = gpd.sjoin_nearest(
        points_gdf.set_geometry("geometry"),
        nodes_reset.set_geometry("geometry"),
        how="left",
        distance_col="dist"
    )
    out = nn[["a_id", "osmid", "dist"]].rename(columns={"osmid": "node"}).assign(label=label_col)

    save_obj(out, cached_name)
    return out

def anchor_cells_to_graph(nodes_gdf, grid_gdf) -> pd.DataFrame:
    cached = load_obj("anchor_demand_cells")
    if cached is not None:
        return cached

    centroids = grid_gdf.copy()
    centroids["geometry"] = centroids.geometry.centroid
    nodes_reset = nodes_gdf.reset_index()
    nn = gpd.sjoin_nearest(centroids, nodes_reset[["osmid","geometry"]], how="left", distance_col="dist")
    out = nn[["d_id","osmid","dist"]].rename(columns={"osmid":"node"})

    save_obj(out, "anchor_demand_cells")
    return out

# -----------------------------------------------------------------------------
# BRANDES-STYLE DEB (subset sources/targets, pair-weighted)
# -----------------------------------------------------------------------------
def deb_edge_brandes_subset(
    G: nx.MultiDiGraph,
    sources: Iterable[int],
    source_w: Dict[int,float],
    target_w: Dict[int,float],
    weight_key: str = "tt",
) -> Dict[Tuple[int,int,int], float]:
    e_c = defaultdict(float)
    tol = 1e-9

    for s in tqdm(list(sources), desc="DEB sources"):
        S = []
        P = defaultdict(list)
        sigma = defaultdict(float)
        dist = defaultdict(lambda: math.inf)

        sigma[s] = 1.0
        dist[s] = 0.0
        import heapq
        Q = [(0.0, s)]

        while Q:
            dv, v = heapq.heappop(Q)
            if dv > dist[v] + 1e-12:
                continue
            S.append(v)
            for w in G.successors(v):
                min_w = math.inf
                for k, data in G[v][w].items():
                    wtt = float(data.get(weight_key, 1.0))
                    if wtt < min_w:
                        min_w = wtt
                vw = dist[v] + min_w
                if vw < dist[w] - tol:
                    dist[w] = vw
                    heapq.heappush(Q, (vw, w))
                    sigma[w] = sigma[v]
                    P[w] = [v]
                elif abs(vw - dist[w]) <= tol:
                    sigma[w] += sigma[v]
                    P[w].append(v)

        delta = defaultdict(float)
        sw = float(source_w.get(s, 1.0))
        while S:
            w = S.pop()
            coef = float(target_w.get(w, 0.0))
            for v in P.get(w, []):
                if sigma[w] == 0:
                    continue
                contrib = (sigma[v] / max(sigma[w], EPS_DENOM)) * (coef + delta[w])

                # split across parallel edges (v,w) that lie on a shortest v->w step
                candidates = []
                if v in G and w in G[v]:
                    for k, data in G[v][w].items():
                        wtt = float(data.get(weight_key, 1.0))
                        if abs(dist[v] + wtt - dist[w]) <= tol:
                            candidates.append((v,w,k))
                m = max(1, len(candidates))
                for edge in candidates:
                    e_c[edge] += sw * contrib / m
                delta[v] += contrib
    return e_c

# -----------------------------------------------------------------------------
# SERVC (two orientations) + prox curb friction
# -----------------------------------------------------------------------------
def node_incident_phi_mean(G: nx.MultiDiGraph, node: int) -> float:
    vals = []
    for _, _, _, d in G.out_edges(node, keys=True, data=True):
        vals.append(float(d.get("phi", 0.0)))
    for _, _, _, d in G.in_edges(node, keys=True, data=True):
        vals.append(float(d.get("phi", 0.0)))
    return float(np.mean(vals)) if vals else 0.0

def shortest_times_from(G: nx.MultiDiGraph, source: int, weight_key="tt") -> Dict[int,float]:
    return nx.single_source_dijkstra_path_length(G, source, weight=weight_key)

def servc_access(
    G: nx.MultiDiGraph,
    access_nodes: Dict[int,int],  # a_id -> node
    demand_nodes: Dict[int,int],  # d_id -> node
    w_d: Dict[int,float],
    orientation: str,             # "AD" or "DA"
    kappa_seconds: Optional[float],
) -> pd.DataFrame:
    # Cached by orientation hash
    cache_name = f"servc_{orientation.lower()}"
    cached = load_obj(cache_name)
    if cached is not None:
        return cached

    rows = []
    edge_ff = [float(d.get("travel_time", 0.0)) for _,_,_,d in G.edges(keys=True, data=True)]
    tau_ref = float(np.median(edge_ff)) if edge_ff else 5.0
    # Stable kappa: at least 1 second
    if kappa_seconds is None or not np.isfinite(kappa_seconds) or kappa_seconds <= 0:
        kappa = max(SERVC_DELTA * tau_ref, 1.0)
    else:
        kappa = max(float(kappa_seconds), 1.0)

    if orientation.upper() == "AD":
        for a_id, node in tqdm(access_nodes.items(), desc="ServC AD (access->demand)"):
            phi_prox = node_incident_phi_mean(G, node)
            dist = shortest_times_from(G, node, weight_key="tt")
            tot = 0.0
            for d_id, dn in demand_nodes.items():
                t = float(dist.get(dn, np.inf))
                if np.isfinite(t):
                    denom = max(t + kappa*phi_prox, EPS_DENOM)
                    tot += (float(w_d.get(d_id, 1.0)) / denom)
            rows.append({"a_id": a_id, "servc": tot, "phi_prox": phi_prox})
    else:
        Gr = G.reverse(copy=False)
        for a_id, node in tqdm(access_nodes.items(), desc="ServC DA (demand->access) via reverse"):
            phi_prox = node_incident_phi_mean(G, node)
            dist = shortest_times_from(Gr, node, weight_key="tt")
            tot = 0.0
            for d_id, dn in demand_nodes.items():
                t = float(dist.get(dn, np.inf))
                if np.isfinite(t):
                    denom = max(t + kappa*phi_prox, EPS_DENOM)
                    tot += (float(w_d.get(d_id, 1.0)) / denom)
            rows.append({"a_id": a_id, "servc": tot, "phi_prox": phi_prox})

    out = pd.DataFrame(rows)
    save_obj(out, cache_name)
    return out

# -----------------------------------------------------------------------------
# OUTCOMES
# -----------------------------------------------------------------------------
def fetch_311_illegal_parking() -> gpd.GeoDataFrame:
    cached = load_obj("g311_illegal_parking")
    if cached is not None:
        return cached

    url = "https://data.cityofnewyork.us/resource/erm2-nwe9.json"
    where = []
    where.append(f"created_date between '{DATE_FROM}T00:00:00.000' and '{DATE_TO}T23:59:59.999'")
    where.append("complaint_type = 'Illegal Parking'")
    where.append("latitude IS NOT NULL AND longitude IS NOT NULL")
    params = {
        "$select": "created_date, latitude, longitude",
        "$where": " AND ".join(where),
    }
    rows = socrata_get(url, params)
    if not rows:
        gdf = gpd.GeoDataFrame(geometry=[], crs=4326).to_crs(3857)
        save_obj(gdf, "g311_illegal_parking")
        return gdf

    df = pd.DataFrame(rows)
    df["created_date"] = pd.to_datetime(df["created_date"], errors="coerce")
    df = df.dropna(subset=["created_date","latitude","longitude"])
    df = df[(df["created_date"].dt.weekday < 5) &
            (df["created_date"].dt.hour >= BUSINESS_HOURS[0]) &
            (df["created_date"].dt.hour <  BUSINESS_HOURS[1])]

    gdf = gpd.GeoDataFrame(df,
        geometry=gpd.points_from_xy(df["longitude"].astype(float),
                                    df["latitude"].astype(float)),
        crs=4326).to_crs(3857)
    save_obj(gdf, "g311_illegal_parking")
    return gdf

def fetch_truck_crashes() -> gpd.GeoDataFrame:
    """
    Robust pull for truck-involved crashes:
    - Query by date + lat/lon only (no brittle SoQL on vehicle columns)
    - Then flag trucks client-side using any vehicle_type_code* column
    - Business-hours + weekday filter applied locally
    - Cached to cache_nyc_ulmm/g_truck_crashes.pkl
    """
    cached = load_obj("g_truck_crashes")
    if cached is not None:
        return cached

    url = "https://data.cityofnewyork.us/resource/h9gi-nx95.json"

    # Try timestamped bounds first, then fall back to date-only if the API complains.
    base_where = (
        f"latitude IS NOT NULL AND longitude IS NOT NULL AND "
        f"crash_date between '{DATE_FROM}T00:00:00.000' and '{DATE_TO}T23:59:59.999'"
    )
    params = {"$where": base_where}

    def _safe_get(p):
        try:
            return socrata_get(url, p)
        except requests.HTTPError:
            # Fallback: date-only bounds (some Socrata backends prefer this for calendar_date)
            p2 = {"$where": f"latitude IS NOT NULL AND longitude IS NOT NULL AND "
                            f"crash_date between '{DATE_FROM}' and '{DATE_TO}'"}
            return socrata_get(url, p2)

    rows = _safe_get(params)
    if not rows:
        gdf_empty = gpd.GeoDataFrame(geometry=[], crs=4326).to_crs(3857)
        save_obj(gdf_empty, "g_truck_crashes")
        return gdf_empty

    df = pd.DataFrame(rows)

    # Find any available vehicle type columns (the dataset sometimes varies)
    veh_cols = [c for c in df.columns if c.lower().startswith("vehicle_type_code")]
    if veh_cols:
        # Normalize strings and flag truck-like entries
        # Covers TRUCK, PICK-UP TRUCK, BOX TRUCK, TRACTOR(TRAILER), COMMERCIAL, etc.
        pat = r"(TRUCK|TRACTOR|COMMERCIAL)"
        mask = df[veh_cols].apply(lambda col: col.astype(str).str.upper().str.contains(pat, regex=True, na=False))
        df = df[mask.any(axis=1)]
    else:
        # If vehicle type columns are absent (rare), keep nothing rather than overcount
        df = df.iloc[0:0]

    if df.empty:
        gdf_empty = gpd.GeoDataFrame(geometry=[], crs=4326).to_crs(3857)
        save_obj(gdf_empty, "g_truck_crashes")
        return gdf_empty

    # Parse timestamp: prefer crash_date + crash_time; fallback to crash_date only
    if "crash_time" in df.columns:
        ts = pd.to_datetime(df["crash_date"].astype(str) + " " + df["crash_time"].astype(str), errors="coerce")
    else:
        ts = pd.to_datetime(df["crash_date"], errors="coerce")

    df = df.assign(dt=ts).dropna(subset=["dt", "latitude", "longitude"])

    # Weekday + business hours
    df = df[(df["dt"].dt.weekday < 5) &
            (df["dt"].dt.hour >= BUSINESS_HOURS[0]) &
            (df["dt"].dt.hour <  BUSINESS_HOURS[1])]

    if df.empty:
        gdf_empty = gpd.GeoDataFrame(geometry=[], crs=4326).to_crs(3857)
        save_obj(gdf_empty, "g_truck_crashes")
        return gdf_empty

    gdf = gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(df["longitude"].astype(float), df["latitude"].astype(float)),
        crs=4326
    ).to_crs(3857)

    save_obj(gdf, "g_truck_crashes")
    return gdf

def fetch_parking_tickets_geocoded() -> gpd.GeoDataFrame:
    cached = load_obj("g_tickets")
    if cached is not None:
        return cached
    gdf = gpd.GeoDataFrame(geometry=[], crs=3857)
    save_obj(gdf, "g_tickets")
    return gdf

# -----------------------------------------------------------------------------
# MAP EVENTS TO EDGES & ACCESS BUFFERS
# -----------------------------------------------------------------------------
def edge_midpoints(edges_gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    cached = load_obj("edges_midpoints")
    if cached is not None:
        return cached

    mids = []
    for _, row in edges_gdf.iterrows():
        geom = row.geometry
        if geom is None or geom.is_empty:
            mids.append(Point(np.nan, np.nan))
        else:
            mids.append(geom.interpolate(0.5, normalized=True))
    out = edges_gdf[["u","v","key"]].copy()
    out["geometry"] = mids
    out = gpd.GeoDataFrame(out, geometry="geometry", crs=edges_gdf.crs)
    out["length_m"] = edges_gdf.geometry.length.values

    save_obj(out, "edges_midpoints")
    return out

def map_points_to_edges(points: gpd.GeoDataFrame, edges_mid_gdf: gpd.GeoDataFrame, label: str) -> pd.Series:
    cache_name = f"counts_{label}"
    cached = load_obj(cache_name)
    if cached is not None:
        return cached

    if points.empty:
        s = pd.Series(dtype=int)
        save_obj(s, cache_name)
        return s

    joined = gpd.sjoin_nearest(points, edges_mid_gdf, how="left", distance_col="dist")
    joined = joined[joined["dist"] <= EVENT_TO_EDGE_MAXDIST_M]
    counts = joined.groupby(["u","v","key"]).size()

    save_obj(counts, cache_name)
    return counts

def buffer_counts_near_access(points: gpd.GeoDataFrame, access_gdf: gpd.GeoDataFrame, cache_name: str) -> pd.DataFrame:
    cached = load_obj(cache_name)
    if cached is not None:
        return cached

    if points.empty or access_gdf.empty:
        out = pd.DataFrame({"a_id": [], "count": []})
        save_obj(out, cache_name)
        return out

    buf = access_gdf.copy()
    buf["geometry"] = buf.geometry.buffer(ACCESS_ACTIVITY_RADIUS_M)
    joined = gpd.sjoin(points, buf[["a_id","geometry"]], how="inner", predicate="within")
    counts = joined.groupby("a_id").size().rename("count").reset_index()

    save_obj(counts, cache_name)
    return counts

# -----------------------------------------------------------------------------
# NEIGHBORHOOD / TRACT FIXED EFFECTS
# -----------------------------------------------------------------------------
def fetch_tracts_2020() -> gpd.GeoDataFrame:
    cached = load_obj("census_tracts_2020")
    if cached is not None:
        return cached

    url = "https://data.cityofnewyork.us/resource/imfq-nf3j.geojson"
    params = {"$select": "geoid,nta2020,cdta2020,boro_name,geom"}
    try:
        rows = socrata_get(url, params)
        resp = requests.get(url, timeout=120)
        resp.raise_for_status()
        gj = resp.json()
        tracts = gpd.GeoDataFrame.from_features(gj["features"], crs=4326).to_crs(3857)
    except Exception:
        resp = requests.get(url, timeout=120)
        resp.raise_for_status()
        gj = resp.json()
        tracts = gpd.GeoDataFrame.from_features(gj["features"], crs=4326).to_crs(3857)

    for f in ["geoid","nta2020","cdta2020","boro_name"]:
        if f not in tracts.columns:
            tracts[f] = None

    tracts = tracts[["geoid","nta2020","cdta2020","boro_name","geometry"]]
    save_obj(tracts, "census_tracts_2020")
    return tracts

# -----------------------------------------------------------------------------
# EVALUATION METRICS
# -----------------------------------------------------------------------------
def top_decile_lift(score: pd.Series, label: pd.Series) -> float:
    n = len(score)
    if n == 0:
        return float("nan")
    k = max(1, int(math.ceil(0.10 * n)))
    idx = score.sort_values(ascending=False).index[:k]
    top_pos_rate = label.loc[idx].mean()
    base_rate = label.mean()
    return (top_pos_rate / base_rate) if base_rate > 0 else float("inf")

def zscore(x: pd.Series) -> pd.Series:
    mu, sd = x.mean(), x.std(ddof=0)
    sd = sd if sd > 0 else 1.0
    return (x - mu) / sd

def fit_nb_irr(df: pd.DataFrame, y: str, x: str,
               fe: Optional[str], offset: Optional[str],
               controls: Optional[List[str]] = None) -> float:
    """
    Negative binomial GLM with log link; return IRR for coefficient on `x`.
    Defensively:
      - Coerces all regressors to numeric
      - Builds FE as float dummies with drop_first
      - Sanitizes NaN/Inf to finite values
      - Adds constant robustly
      - Returns NaN on failure instead of raising
    """
    df_use = df.copy()

    # 1) Coerce numeric columns (main regressor + controls)
    reg_cols = [x] + (controls or [])
    for col in reg_cols:
        df_use[col] = pd.to_numeric(df_use[col], errors="coerce")

    # 2) Fixed effects -> dummies (float), drop_first to avoid perfect collinearity
    if fe:
        fe_ser = df_use[fe].astype(str).fillna("NA")
        fe_dum = pd.get_dummies(fe_ser, prefix=fe, drop_first=True, dtype=float)
        X = pd.concat([df_use[reg_cols], fe_dum], axis=1)
    else:
        X = df_use[reg_cols]

    # 3) Clean & cast design matrix to float
    X = X.replace([np.inf, -np.inf], np.nan).fillna(0.0).astype(float)

    # 4) Endog (counts) as non-negative integers
    y_arr = pd.to_numeric(df_use[y], errors="coerce").fillna(0).clip(lower=0).astype(np.int64).values

    # 5) Offset (log-length etc.) if provided
    if offset:
        off_raw = pd.to_numeric(df_use[offset], errors="coerce")
        off_raw = off_raw.replace([np.inf, -np.inf], np.nan).fillna(0.0).clip(lower=1e-6)
        off = np.log(off_raw.values.astype(float))
    else:
        off = None

    # 6) Add constant and fit
    X = sm.add_constant(X, has_constant="add")

    try:
        model = GLM(y_arr, X, family=NegativeBinomial(), offset=off)
        result = model.fit(maxiter=200, disp=0)
        coef = result.params.get(x, np.nan)
        return float(np.exp(coef)) if np.isfinite(coef) else float("nan")
    except Exception as e:
        print(f"[warn] NB GLM failed: {e}")
        return float("nan")

# -----------------------------------------------------------------------------
# MAIN
# -----------------------------------------------------------------------------
def main():
    t0 = time.time()
    print(">> Building NYC ULMM core and computing scores…")

    # Boundary
    nyc_poly_wgs84 = get_nyc_boundary_wgs84()
    nyc_poly_3857  = nyc_poly_wgs84.to_crs(3857)

    # Graph
    G, nodes_gdf, edges_gdf = build_curb_aware_graph(nyc_poly_wgs84)

    # Midpoints (segments layer)
    edges_mid = edge_midpoints(edges_gdf)

    # Demand grid + POIs + weights
    grid = build_demand_grid(nyc_poly_3857)
    pois = download_osm_pois(nyc_poly_wgs84)
    demand = weight_demand_cells(grid, pois).reset_index(drop=True)

    # Access points
    access = download_access_points(nyc_poly_wgs84)
    if access.empty:
        print("WARNING: No access points found after fallbacks; ServC metrics will be skipped.")

    # Anchor nodes
    acc_nn = anchor_points_to_graph(nodes_gdf, access, label_col="access")
    dem_nn = anchor_cells_to_graph(nodes_gdf, demand)

    access_nodes = dict(zip(acc_nn.get("a_id", []), acc_nn.get("node", [])))
    demand_nodes = dict(zip(dem_nn["d_id"], dem_nn["node"]))
    w_d = dict(zip(demand["d_id"], demand["w_d"]))
    alpha_a = dict(zip(access.get("a_id", []), access.get("alpha", [])))

    # --- DEB (AD orientation) ---
    deb_df = load_obj("deb_df")
    if deb_df is None:
        target_weight_by_node = defaultdict(float)
        for d_id, n in demand_nodes.items():
            target_weight_by_node[n] += float(w_d.get(d_id, 1.0))

        source_weight_by_node = defaultdict(float)
        for a_id, n in access_nodes.items():
            source_weight_by_node[n] += float(alpha_a.get(a_id, 1.0))

        deb = deb_edge_brandes_subset(
            G,
            sources=list(source_weight_by_node.keys()),
            source_w=source_weight_by_node,
            target_w=target_weight_by_node,
            weight_key="tt",
        )
        deb_df = pd.DataFrame(
            [(u,v,k,val) for (u,v,k), val in deb.items()],
            columns=["u","v","key","DEB"]
        )
        save_obj(deb_df, "deb_df")

    edges_scored = load_obj("edges_scored")
    if edges_scored is None:
        edges_scored = edges_gdf.merge(deb_df, on=["u","v","key"], how="left").fillna({"DEB": 0.0})
        edges_scored["DEB_z"] = zscore(edges_scored["DEB"])
        save_obj(edges_scored, "edges_scored")

    # --- ServC (kappa scaled by city edge median ff-time; min 1s) ---
    try:
        tt = edges_gdf["travel_time"].astype(float).replace([np.inf, -np.inf], np.nan).dropna().values
        ff_median = float(np.median(tt)) if tt.size else 5.0
    except Exception:
        ff_median = 5.0
    if not np.isfinite(ff_median) or ff_median <= 0:
        ff_median = 5.0
    kappa = max(SERVC_DELTA * ff_median, 1.0)

    servc_ad = pd.DataFrame()
    servc_da = pd.DataFrame()
    if not access.empty and len(access_nodes) > 0:
        servc_ad = servc_access(G, access_nodes, demand_nodes, w_d, orientation="AD", kappa_seconds=kappa)
        servc_da = servc_access(G, access_nodes, demand_nodes, w_d, orientation="DA", kappa_seconds=kappa)
        if not servc_ad.empty:
            servc_ad["ServC_z"] = zscore(servc_ad["servc"])
        if not servc_da.empty:
            servc_da["ServC_z"] = zscore(servc_da["servc"])

    # --- Held-out outcomes ---
    print(">> Downloading held-out curb activity…")
    g311 = fetch_311_illegal_parking()
    gtr  = fetch_truck_crashes()
    gtix = fetch_parking_tickets_geocoded()  # empty unless plugged

    # Map to edges
    print(">> Mapping events to street segments…")
    c_311 = map_points_to_edges(g311, edges_mid, label="311")
    c_trk = map_points_to_edges(gtr,  edges_mid, label="truck")
    c_tix = map_points_to_edges(gtix, edges_mid, label="tickets") if not gtix.empty else pd.Series(dtype=int)

    seg = load_obj("segments_table")
    if seg is None:
        seg = edges_mid.merge(c_311.rename("n_311"), on=["u","v","key"], how="left")
        seg = seg.merge(c_trk.rename("n_truck"), on=["u","v","key"], how="left")
        if not c_tix.empty:
            seg = seg.merge(c_tix.rename("n_tickets"), on=["u","v","key"], how="left")
        seg = seg.fillna({"n_311": 0, "n_truck": 0, "n_tickets": 0})
        seg = seg.merge(edges_scored[["u","v","key","DEB_z"]], on=["u","v","key"], how="left")

        # retail density near edges (50 m buffer)
        if not pois.empty:
            edge_buf = edges_gdf[["u","v","key","geometry"]].copy()
            edge_buf["geometry"] = edge_buf.geometry.buffer(50)
            j = gpd.sjoin(pois[["geometry"]], edge_buf, how="inner", predicate="within")
            cnt = j.groupby(["u","v","key"]).size().rename("n_poi50").reset_index()
            seg = seg.merge(cnt, on=["u","v","key"], how="left").fillna({"n_poi50": 0})
        else:
            seg["n_poi50"] = 0

        save_obj(seg, "segments_table")

    # Hotspot labels
    def make_hotspot(s: pd.Series) -> pd.Series:
        if (s>0).sum() == 0:
            return pd.Series(np.zeros(len(s), dtype=int), index=s.index)
        thr = np.quantile(s[s>0], 0.90)
        return (s >= thr).astype(int)

    seg["y_311_hot"]  = make_hotspot(seg["n_311"])
    seg["y_trk_hot"]  = make_hotspot(seg["n_truck"])
    if "n_tickets" in seg.columns and seg["n_tickets"].sum() > 0:
        seg["y_tix_hot"] = make_hotspot(seg["n_tickets"])
    else:
        seg["y_tix_hot"] = make_hotspot(seg["n_311"])

    # Join tract FE for segments (use midpoints)
    print(">> Joining census tracts (FE)…")
    tracts = fetch_tracts_2020()
    seg_fe = load_obj("segments_fe")
    if seg_fe is None:
        seg_g = gpd.GeoDataFrame(seg, geometry="geometry", crs=3857)
        seg_fe = gpd.sjoin(seg_g, tracts[["geoid","nta2020","cdta2020","geometry"]],
                           how="left", predicate="within").drop(columns=["index_right"])
        save_obj(seg_fe, "segments_fe")

    # Access-level activity & FE (only if access exists)
    acc_fe = pd.DataFrame()
    if not access.empty:
        acc = access.copy()

        if not servc_ad.empty:
            acc = acc.merge(
                servc_ad[["a_id","ServC_z","phi_prox"]].rename(
                    columns={"ServC_z":"ServC_AD_z","phi_prox":"phi_prox_ad"}),
                on="a_id", how="left")
        else:
            acc["ServC_AD_z"] = np.nan
            acc["phi_prox_ad"] = np.nan

        if not servc_da.empty:
            acc = acc.merge(
                servc_da[["a_id","ServC_z","phi_prox"]].rename(
                    columns={"ServC_z":"ServC_DA_z","phi_prox":"phi_prox_da"}),
                on="a_id", how="left")
        else:
            acc["ServC_DA_z"] = np.nan
            acc["phi_prox_da"] = np.nan

        c_acc_311 = buffer_counts_near_access(g311, acc[["a_id","geometry"]], cache_name="counts_access_311")
        c_acc_tix = buffer_counts_near_access(gtix, acc[["a_id","geometry"]], cache_name="counts_access_tix") if not gtix.empty else pd.DataFrame({"a_id": [], "count": []})

        acc = acc.merge(c_acc_311.rename(columns={"count":"n_311_50"}), on="a_id", how="left").fillna({"n_311_50": 0})
        if not c_acc_tix.empty:
            acc = acc.merge(c_acc_tix.rename(columns={"count":"n_tix_50"}), on="a_id", how="left").fillna({"n_tix_50": 0})
        else:
            acc["n_tix_50"] = 0
        acc["n_act_50"] = acc["n_311_50"] + acc["n_tix_50"]

        if (acc["n_act_50"]>0).sum() > 0:
            thr_acc = np.quantile(acc.loc[acc["n_act_50"]>0, "n_act_50"], 0.90)
        else:
            thr_acc = np.inf
        acc["y_acc_hot"] = (acc["n_act_50"] >= thr_acc).astype(int)

        acc_fe = load_obj("access_fe")
        if acc_fe is None:
            acc_fe = gpd.sjoin(acc, tracts[["geoid","nta2020","cdta2020","geometry"]],
                               how="left", predicate="within").drop(columns=["index_right"])
            save_obj(acc_fe, "access_fe")

    # -------------------------------------------------------------------------
    # METRICS
    # -------------------------------------------------------------------------
    print(">> Computing metrics & NB IRRs…")

    def seg_metrics(ycol):
        y = seg_fe[ycol].astype(int)
        s = seg_fe["DEB_z"].astype(float)
        auc = roc_auc_score(y, s) if y.nunique() > 1 else float("nan")
        lift = top_decile_lift(s, y)

        # NB IRR with tract FE & length offset & retail density (as control)
        count_map = {"y_tix_hot":"n_311" if "n_tickets" not in seg_fe else "n_tickets",
                     "y_311_hot":"n_311",
                     "y_trk_hot":"n_truck"}
        df_nb = pd.DataFrame({
            "y": seg_fe[count_map[ycol]].astype(int).values,
            "DEB_z": seg_fe["DEB_z"].values,
            "geoid": seg_fe["geoid"].astype(str).fillna("NA").values,
            "length_m": seg_fe["length_m"].values if "length_m" in seg_fe.columns else seg_fe["geometry"].length.values,
            "retail": seg_fe["n_poi50"].values
        })
        irr = fit_nb_irr(df_nb, y="y", x="DEB_z", fe="geoid", offset="length_m", controls=["retail"])
        return auc, lift, irr

    auc_tix = load_obj("auc_tix")
    lift_tix = load_obj("lift_tix")
    irr_tix = load_obj("irr_tix")
    auc_311 = load_obj("auc_311")
    lift_311 = load_obj("lift_311")
    irr_311 = load_obj("irr_311")
    auc_trk = load_obj("auc_trk")
    lift_trk = load_obj("lift_trk")
    irr_trk = load_obj("irr_trk")

    if auc_tix is None:
        auc_tix, lift_tix, irr_tix = seg_metrics("y_tix_hot")
        save_obj(auc_tix, "auc_tix")
        save_obj(lift_tix, "lift_tix")
        save_obj(irr_tix, "irr_tix")

    if auc_311 is None:
        auc_311, lift_311, irr_311 = seg_metrics("y_311_hot")
        save_obj(lift_311, "lift_311")
        save_obj(irr_311, "irr_311")
        save_obj(auc_311, "auc_311")

    if auc_trk is None:
        auc_trk, lift_trk, irr_trk = seg_metrics("y_trk_hot")
        save_obj(auc_trk, "auc_trk")
        save_obj(lift_trk, "lift_trk")
        save_obj(irr_trk, "irr_trk")
    
    print(">> Done first")

    def access_metrics(score_col):
        if acc_fe.empty:
            return (float("nan"), float("nan"), float("nan"))
        y = acc_fe["y_acc_hot"].astype(int)
        s = acc_fe[score_col].astype(float)
        auc = roc_auc_score(y, s) if y.nunique() > 1 else float("nan")
        lift = top_decile_lift(s, y)
        df_nb = pd.DataFrame({
            "y": acc_fe["n_act_50"].astype(int).values,
            "x": acc_fe[score_col].values,
            "phi": acc_fe["phi_prox_ad"].fillna(acc_fe["phi_prox_da"]).fillna(0).values,
            "geoid": acc_fe["geoid"].astype(str).fillna("NA").values
        })
        irr = fit_nb_irr(df_nb, y="y", x="x", fe="geoid", offset=None, controls=["phi"])
        return auc, lift, irr

    auc_acc_ad = load_obj("auc_acc_ad")
    lift_acc_ad = load_obj("lift_acc_ad")
    irr_acc_ad = load_obj("irr_acc_ad")
    auc_acc_da = load_obj("auc_acc_da")
    lift_acc_da = load_obj("lift_acc_da")
    irr_acc_da = load_obj("irr_acc_da")

    if auc_acc_ad is None:
        auc_acc_ad, lift_acc_ad, irr_acc_ad = access_metrics("ServC_AD_z")
        save_obj(auc_acc_ad, "auc_acc_ad")
        save_obj(lift_acc_ad, "lift_acc_ad")
        save_obj(irr_acc_ad, "irr_acc_ad")

    if auc_acc_da is None:
        auc_acc_da, lift_acc_da, irr_acc_da = access_metrics("ServC_DA_z")
        save_obj(auc_acc_da, "auc_acc_da")
        save_obj(lift_acc_da, "lift_acc_da")
        save_obj(irr_acc_da, "irr_acc_da")

    print(">> Done second")

    metrics = {
        "segments": {
            "violations": {"auc": auc_tix, "lift": lift_tix, "irr": irr_tix},
            "311":        {"auc": auc_311, "lift": lift_311, "irr": irr_311},
            "truck":      {"auc": auc_trk, "lift": lift_trk, "irr": irr_trk},
        },
        "access": {
            "AD": {"auc": auc_acc_ad, "lift": lift_acc_ad, "irr": irr_acc_ad},
            "DA": {"auc": auc_acc_da, "lift": lift_acc_da, "irr": irr_acc_da},
        }
    }
    save_obj(metrics, "metrics_summary")

    print(">> All metrics saved")

    # -------------------------------------------------------------------------
    # TABLE -> LaTeX
    # -------------------------------------------------------------------------
    def fmt(x):
        if x is None or (isinstance(x,float) and (np.isnan(x) or np.isinf(x))):
            return "--"
        return f"{x:.2f}"

    rows = [
        ("Violations hotspot (segments)", r"$z(\mathrm{DEB}_\xi)$", fmt(auc_tix), fmt(lift_tix), fmt(irr_tix)),
        ("311 hotspot (segments)",        r"$z(\mathrm{DEB}_\xi)$", fmt(auc_311), fmt(lift_311), fmt(irr_311)),
        ("Truck-involved crash hotspot",  r"$z(\mathrm{DEB}_\xi)$", fmt(auc_trk), fmt(lift_trk), fmt(irr_trk)),
    ]

    if not access.empty and not servc_ad.empty and not servc_da.empty:
        rows.extend([
            ("High activity near access (50\\,m)", r"$z(\mathrm{ServC}^{\\rightarrow})$", fmt(auc_acc_ad), fmt(lift_acc_ad), fmt(irr_acc_ad)),
            ("High activity near access (50\\,m)", r"$z(\mathrm{ServC}^{\\leftarrow})$",  fmt(auc_acc_da), fmt(lift_acc_da), fmt(irr_acc_da)),
        ])
    else:
        rows.extend([
            ("High activity near access (50\\,m)", r"$z(\mathrm{ServC}^{\\rightarrow})$", "--", "--", "--"),
            ("High activity near access (50\\,m)", r"$z(\mathrm{ServC}^{\\leftarrow})$",  "--", "--", "--"),
        ])

    latex = textwrap.dedent(f"""
    \\begin{{table}}[ht]
    \\centering
    \\scriptsize
    \\caption{{NYC external validation (held-out). Predicting observed curb activity with ULMM scores.}}
    \\label{{tab:external-nyc}}
    \\begin{{tabular}}{{lcccc}}
    \\toprule
    Outcome (NYC) & Score & ROC--AUC & Top-decile Lift & NBIRR$^\\dagger$ \\\\
    \\midrule
    {" \\\\\n".join([f"{a} & {b} & {c} & {d} & {e}" for (a,b,c,d,e) in rows])}
    \\\\ \\bottomrule
    \\end{{tabular}}
    \\end{{table}}
    """).strip()

    out_path = "table_nyc_external_validation.tex"
    with open(out_path, "w", encoding="utf-8") as f:
        f.write(latex)

    print("\nLaTeX table written to:", out_path)
    print("\nPreview:\n", latex)
    print(f"\nTotal runtime: {time.time()-t0:.1f}s")

In [2]:
main()

>> Building NYC ULMM core and computing scores…
[cache] loaded <- cache_nyc_ulmm/nyc_boundary_wgs84.pkl
[cache] loaded <- cache_nyc_ulmm/graph_G.pkl
[cache] loaded <- cache_nyc_ulmm/graph_nodes_gdf.pkl
[cache] loaded <- cache_nyc_ulmm/graph_edges_gdf.pkl
[cache] loaded <- cache_nyc_ulmm/edges_midpoints.pkl
[cache] loaded <- cache_nyc_ulmm/demand_grid.pkl
[cache] loaded <- cache_nyc_ulmm/pois.pkl
[cache] loaded <- cache_nyc_ulmm/demand_weighted.pkl
[cache] loaded <- cache_nyc_ulmm/access_points.pkl
[cache] loaded <- cache_nyc_ulmm/anchor_access.pkl
[cache] loaded <- cache_nyc_ulmm/anchor_demand_cells.pkl
[cache] loaded <- cache_nyc_ulmm/deb_df.pkl
[cache] loaded <- cache_nyc_ulmm/edges_scored.pkl
[cache] loaded <- cache_nyc_ulmm/servc_ad.pkl
[cache] loaded <- cache_nyc_ulmm/servc_da.pkl
>> Downloading held-out curb activity…
[cache] loaded <- cache_nyc_ulmm/g311_illegal_parking.pkl
[cache] loaded <- cache_nyc_ulmm/g_truck_crashes.pkl
[cache] loaded <- cache_nyc_ulmm/g_tickets.pkl
>> Map